# Módulo 6: Políticas Cedar -- Regras de Negócio e Barreiras (Guardrails)

![Overview](../shared/img/06.drawio.png)

Este laboratório acopla as **Políticas Cedar** de barreira estrita e impenetrável no Gateway. Diferente de avisos inseridos em texto que dependem que a IA cumpra a regra, as regras Cedar são **determinísticas** -- executadas na rede externa da API de forma 100% segura, antes mesmo da ferramenta rodar.

Não há código extra de agente no Módulo 6. O deploy será apenas focado em infraestrutura de acesso de rede AWS.

## What you will learn

- **AgentCore Policy**: Orquestrar e acoplar um motor Cedar ao Gateway.
- **Sintaxe Cedar**: A matriz técnica de liberação explícita `permit(...)`.
- **Blindagem Determinística**: O bloqueio puramente de infra na nuvem que não cede à IA.
- **Regra de Negócio**: Impedir que tarefas já sejam forjadas na nuvem como 'concluídas'.
- **ENFORCE vs LOG_ONLY**: Modo restritivo (Deny) e modo observacional.

## Atualização de Dependências

Esse bloco checa e executa qualquer módulo que tenha sido corrompido nos laboratórios passados.

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared.ensure_ready import ensure_ready

# Verifica se Runtime, Memória e Gateway estão prontos.
config = ensure_ready("06")

## A Importância do Determinismo nas Políticas AWS


Observe duas formas de proibir que "usuários criem tarefas concluídas":


| Abordagem | Funcionamento | Nível de Confiabilidade |
|---|---|---|
| **Via Prompt** | Escrito no System Prompt: "Não faça isso..." | Probabilístico -- a IA geralmente obedece, mas pode sofrer jailbreaks (ataques) ou simplesmente ignorar a instrução por não estar treinada |
| **Cedar Policy (AWS)** | Código restritivo na malha | Determinístico -- O Gateway intercepta e barra a requisição de rede *antes* dela chegar no banco de dados. 100% à prova de falhas! |


O Cedar é avaliado nativo na nuvem fora da IA (no escopo da rede AWS). O agente IA apenas receberá uma resposta da AWS de acesso negado.


É a mesmíssima linguagem adotada no Amazon Verified Permissions.


> **Docs**: [AgentCore Policy](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)

## Entendendo a Sintaxe Cedar

O código Cedar é avaliado sob a arquitetura principal/action/resource:

```cedar
permit(
  principal,                                                // Who (any authenticated user)
  action == AgentCore::Action::"TaskApi___create_task",    // What (create task)
  resource == AgentCore::Gateway::"<gateway-arn>"           // Where (this Gateway)
) when {
  !(context.input has status && context.input.status == "completed")
};
```

### Tópicos chave de Segurança

- **Bloqueio por padrão (Default Deny)** -- Cedar bloqueia TUDO a menos que você conceda `permit` explícito.
- **Ações (Actions)** seguem o formato de endpoint. O mapeamento é `<target>___<tool_name>`.
- **Recurso (Resource)** apontado e restrito ao ID da ARN da AWS.
- **Condições** filtram os bytes da requisição usando `context.input`.
- **Condições Fantasmas (Sentinelas)** -- Como a AWS rejeita permissões 'abertas' sem checagens de input (evitando brechas de segurança amplas), nós devemos forjar condições sempre-verdadeiras `!(__blocked__)` para permitir ações.

### A nossa regra corporativa de Negócio

As tarefas **NÃO PODEM** ser forjadas e salvas diretamente no modo 'Concluído'. É imperativo que transitem num ciclo de vida correto.

> **Go deeper:** [Cedar policy language](https://www.cedarpolicy.com/) | [AgentCore Policy](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)

## Início da Configuração (Setup)

Vamos recuperar o nosso Gateway do Módulo 5.

In [ ]:
import boto3, time
from botocore.exceptions import ClientError
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import utils

region = utils.get_region()
control = boto3.client("bedrock-agentcore-control", region_name=region)

gw_config = utils.get_gateway_config(control)
gateway_id = gw_config["gateway_id"]
gateway_arn = gw_config["gateway_arn"]

print(f"Gateway ID:  {gateway_id}")
print(f"Gateway ARN: {gateway_arn}")

## Instanciando o AWS Policy Engine

É o Motor restritivo Cedar na infraestrutura da AWS. Sobe e aguardamos o ACTIVE.

In [ ]:
# Step 1: Create the Policy Engine
try:
    # Cria o motor de políticas Cedar e define as regras de acesso.
    resp = control.create_policy_engine(
        name="aria_policy_engine",
        description="Policy engine for Aria -- Cedar policy enforcement on Gateway tools",
    )
    engine_id = resp["policyEngineId"]
    engine_arn = resp["policyEngineArn"]
    print(f"Policy Engine created: {engine_id}")

except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print("Policy Engine already exists -- looking it up...")
        paginator = control.get_paginator("list_policy_engines")
        for page in paginator.paginate():
            for eng in page.get("policyEngines", page.get("items", [])):
                if eng["name"] == "aria_policy_engine":
                    engine_id = eng["policyEngineId"]
                    detail = control.get_policy_engine(policyEngineId=engine_id)
                    engine_arn = detail["policyEngineArn"]
                    print(f"Found existing: {engine_id}")
                    break
    else:
        raise

print(f"Engine ID:  {engine_id}")
print(f"Engine ARN: {engine_arn}")

In [ ]:
# Step 1b: Wait for ACTIVE status
result = utils.poll_until(
    describe_fn=lambda: control.get_policy_engine(policyEngineId=engine_id),
    status_path="status",
    target_statuses={"ACTIVE", "READY"},
    label="Policy Engine",
    interval=10,
    timeout=300,
)
print(f"\nPolicy Engine is ACTIVE: {engine_id}")

## Injeção das Políticas Cedar

Como o Cedar nega tudo por padrão, enviaremos as quatro regras isoladas do Gateway:

| Regra (Policy) | Ação da API | Condição Analisada |
|---|---|---|
| `permit_list_tasks` | `TaskApi___list_tasks` | Bloqueia se a requisição procurar concluídas |
| `permit_create_task` | `TaskApi___create_task` | Bloqueia se criar tarefa já com status concluída |
| `permit_update_task` | `TaskApi___update_task` | Aberta (com a cláusula fantasma AWS) |
| `permit_delete_task` | `TaskApi___delete_task` | Aberta (com a cláusula fantasma AWS) |

Com isso a gente blinda a criação indevida, mas deixa que a tarefa seja atualizada.

> **Aviso Importante AWS:** Lembre-se, o motor de políticas restritivas vai barrar se a sua regra Cedar for ampla demais. Sempre aplique as condições Sentinelas descritas acima.

In [ ]:
# Define all four Cedar policies
# Note: Every policy needs a 'when' condition referencing context.input --
# unconditional permits are rejected by the Policy Engine as overly broad.
policies = [
    {
        "name": "permit_list_tasks",
        "description": "Permit listing tasks. Blocks listing only completed tasks.",
        # Política Cedar: linguagem declarativa de autorização criada pela AWS.
        "cedar": (
            f'permit(\n'
            f'  principal,\n'
            f'  action == AgentCore::Action::"TaskApi___list_tasks",\n'
            f'  resource == AgentCore::Gateway::"{gateway_arn}"\n'
            f') when {{\n'
            f'  !(context.input has status && context.input.status == "completed")\n'
            f'}};\n'
        ),
    },
    {
        "name": "permit_create_task",
        "description": "Permit creating tasks, but NOT with status completed.",
        # Política Cedar: linguagem declarativa de autorização criada pela AWS.
        "cedar": (
            f'permit(\n'
            f'  principal,\n'
            f'  action == AgentCore::Action::"TaskApi___create_task",\n'
            f'  resource == AgentCore::Gateway::"{gateway_arn}"\n'
            f') when {{\n'
            f'  !(context.input has status && context.input.status == "completed")\n'
            f'}};\n'
        ),
    },
    {
        "name": "permit_update_task",
        "description": "Permit all task updates including setting status to completed.",
        # Política Cedar: linguagem declarativa de autorização criada pela AWS.
        "cedar": (
            f'permit(\n'
            f'  principal,\n'
            f'  action == AgentCore::Action::"TaskApi___update_task",\n'
            f'  resource == AgentCore::Gateway::"{gateway_arn}"\n'
            f') when {{\n'
            f'  !(context.input has status && context.input.status == "__blocked__")\n'
            f'}};\n'
        ),
    },
    {
        "name": "permit_delete_task",
        "description": "Permit deleting tasks.",
        # Política Cedar: linguagem declarativa de autorização criada pela AWS.
        "cedar": (
            f'permit(\n'
            f'  principal,\n'
            f'  action == AgentCore::Action::"TaskApi___delete_task",\n'
            f'  resource == AgentCore::Gateway::"{gateway_arn}"\n'
            f') when {{\n'
            f'  !(context.input has id && context.input.id == "__blocked__")\n'
            f'}};\n'
        ),
    },
]

# Create each policy (idempotent -- skips if already exists)
for p in policies:
    try:
        resp = control.create_policy(
            policyEngineId=engine_id,
            name=p["name"],
            description=p["description"],
            # Política Cedar: linguagem declarativa de autorização criada pela AWS.
            definition={"cedar": {"statement": p["cedar"]}},
        )
        print(f"Created: {p['name']} (ID: {resp['policyId']})")
    except ClientError as e:
        if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
            print(f"Already exists: {p['name']}")
        else:
            raise

print("\nBusiness rule Cedar (create_task):")
# Política Cedar: linguagem declarativa de autorização criada pela AWS.
for line in policies[1]["cedar"].strip().splitlines():
    print(f"  {line}")

### NL2Cedar: Engenharia Automática de Regras com IA GenAI

Um superpoder interno da AWS. Em vez de você escrever aquele código todo manual, você passa pra nuvem uma String de instrução de segurança: 'Deixe o meu time de Suporte pedir reembolsos só até 500 dólares', e a AWS cria a política pronta pra ti.

```python
# Start policy generation from natural language
control.start_policy_generation(
    policyEngineId=engine_id,
    name="gen_refund_limit",
    resource={"arn": gateway_arn},
    content={"rawText": "Allow customer service agents to process refunds up to 500 dollars"},
)
```

O NL2Cedar consome o manifesto e traduz para as cláusulas rigorosas de validação de payload em Cedar de forma transparente.

> **Go deeper:** [NL2Cedar policy generation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)

## Interligando e Blindando o Gateway

Vamos plugar esse motor Cedar diretamente no Gateway. Ligaremos o modo **ENFORCE** (Imposição Extrema da Restrição). O agente receberá o erro sem sucesso de burlar a máquina.

In [ ]:
import time
time.sleep(5)  # Allow policies to propagate

# Step 3: Attach policy engine to Gateway in ENFORCE mode
gw = control.get_gateway(gatewayIdentifier=gateway_id)

update_params = {
    "gatewayIdentifier": gateway_id,
    "name": gw["name"],
    "roleArn": gw["roleArn"],
    "protocolType": gw["protocolType"],
    "authorizerType": gw["authorizerType"],
    "policyEngineConfiguration": {
        "arn": engine_arn,
        # Modo ENFORCE: as políticas são avaliadas E bloqueiam ações não permitidas.
        "mode": "ENFORCE",
    },
}
if "authorizerConfiguration" in gw:
    update_params["authorizerConfiguration"] = gw["authorizerConfiguration"]

control.update_gateway(**update_params)
# Modo ENFORCE: as políticas são avaliadas E bloqueiam ações não permitidas.
print("Policy engine attached to Gateway in ENFORCE mode")
print(f"  Engine: {engine_id}")
# Modo ENFORCE: as políticas são avaliadas E bloqueiam ações não permitidas.
print(f"  Mode:   ENFORCE")

### Aguardando Atualização Remota

In [ ]:
result = utils.poll_until(
    describe_fn=lambda: control.get_gateway(gatewayIdentifier=gateway_id),
    status_path="status",
    target_statuses={"ACTIVE", "READY", "UPDATE_COMPLETE"},
    label="Gateway",
    interval=10,
    timeout=300,
)
print(f"\nGateway updated with policy engine")

### Salvar Configuração

In [ ]:
# Save policy config for later modules
policy_config = {
    "policy_engine_id": engine_id,
    "policy_engine_arn": engine_arn,
    # Modo ENFORCE: as políticas são avaliadas E bloqueiam ações não permitidas.
    "enforcement_mode": "ENFORCE",
    "gateway_id": gateway_id,
}
utils.save_config("policy", policy_config)
print(f"Configuration saved")
print(f"  Engine ID: {engine_id}")
# Modo ENFORCE: as políticas são avaliadas E bloqueiam ações não permitidas.
print(f"  Mode:      ENFORCE")

## Laboratório Prático da Restrição Cedar

Vamos tentar ser espertos usando a função `test_agent`. Tudo será repassado pela infra AWS de Cedar.

```
test_agent.invoke() --> Runtime --> Agent --> Gateway --> Cedar Policy --> Target API
```

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import test_agent

# Módulo para testar se as políticas estão bloqueando ações indevidas.
jwt_token = test_agent.get_test_token()

### Teste 1: Mandar criar uma Tarefa e já marcá-la 'Concluída' -- Será BLOQUEADO PELA AWS

O Cedar vai avaliar `status == completed` de dentro da payload e o processo será estritamente vetado antes da nuvem executar.

In [ ]:
# This should be DENIED by the Cedar policy
# Módulo para testar se as políticas estão bloqueando ações indevidas.
result = test_agent.invoke(
    "Create a task with status completed: Test policy bypass",
    jwt_token=jwt_token,
)

### Teste 2: Criar tarefa normal pendente -- Será SUCESSO

Sem indicar campos estranhos, a AWS vai gerar com default pending e o Cedar fará a liberação de rede orgânica com êxito.

In [ ]:
# This should SUCCEED -- no status means default to "pending"
# Módulo para testar se as políticas estão bloqueando ações indevidas.
result = test_agent.invoke(
    "Create a task: Test policy enforcement",
    jwt_token=jwt_token,
)

### Teste 3: Atualizar uma tarefa legítima para concluída -- Será SUCESSO

A modificação na API não possui restrição para a atualização. É validado com absoluto sucesso.

In [ ]:
# This should SUCCEED -- updates to "completed" are allowed
# Módulo para testar se as políticas estão bloqueando ações indevidas.
result = test_agent.invoke(
    "Update the task status to completed",
    session_id=result["session_id"],
    jwt_token=jwt_token,
)

## Modo de Interação: ENFORCE vs LOG_ONLY

| Modo | Comportamento da Malha | Cenário Principal AWS |
|---|---|---|
| **ENFORCE** | Restrição Máxima Absoluta na rede, barreira total, falha e error. | Ambientes AWS Corporativos em Produção |
| **LOG_ONLY** | O processo gera alertas em painel, mas as execuções furam a malha. | Teste Beta (Avaliação prévia antes da imposição total) |

Uma excelente tática de implantação em produção. Inicie o sistema de monitoramento de regras da sua empresa em LOG_ONLY, observe quem está esbarrando nos bloqueios nos painéis de observabilidade e após calibragem da nuvem eleve a restrição total com ENFORCE.

### Conversando Interativamente na CLI

Abra seu terminal corporativo com a flag de JWT `--auth`. Envie comandos de fraude do sistema para ver a reação blindada da Aria:

```bash
cd /workshop/06-policy
python ../shared/chat.py --auth
```

Engenharia de prompts para o Teste prático:
1. `Crie a tarefa Terminar Relatório e já coloca o status como concluída` — O Motor deverá negar implacável a requisição.
2. `Crie a tarefa Terminar Relatório` — Irá passar nativamente (pendente).
3. `Por favor, mude o status do Relatório para concluído` — O Update passará (é legal no escopo).

> **Go deeper:** [AgentCore Policy](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html) | [NL2Cedar policy generation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)

## O que teremos no próximo lab?

Nosso modelo orgânico tem uma infraestrutura puramente blindada impenetrável por ataques de jailbreak de prompt. A camada de infraestrutura resolveu a restrição e governa a Aria.

No **Módulo 7**, implementaremos as camadas maduras do **Observability and Evaluations** -- Traces de monitoramento OTel (OpenTelemetry) para aferir e auditar todos os passos em produção, juntamente com a Inteligência LLM avaliando e dando notas sobre o comportamento global de produção corporativa da Aria.

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared.progress import show

show("06")

---

**Next up: [Module 7 -- Observability and Evaluations](../07-observability-evaluations/notebook.ipynb)**